# E1 subset: 350 nuScenes scenes, full-size CAM_FRONT keyframes

Builds the reduced dataset for the next stage (Meeting 6, Task 2).

| Part | Scenes | Taken from | Used for |
|---|---|---|---|
| boston_c1 | 100 | boston_train (280), even stride | training the old perception module C1 |
| boston_c23 | 100 | boston_val (187), even stride | training C2 and C3 (scenes C1 never saw) |
| singapore_update | 100 | singapore_train (229), even stride | updating C1 on the new city |
| singapore_test | 50 | singapore_val (154), the SAME 50 scenes as the paper | testing |

The split logic reproduces `labsd.splits.partition_by_location` (name-sorted, 60/40 per city) and `cap_val_scenes` (stride 154/50), so the test scenes are identical to the paper.
Output: `/kaggle/working/nuscenes_subset350/` with the full `v1.0-trainval` metadata, only the selected CAM_FRONT keyframe images at full size, and `splits_350.json`.

In [ ]:
import os, json, glob, math, shutil, time
from pathlib import Path
T0 = time.time()
def clock(m): print(f'[{time.time()-T0:7.1f}s] {m}', flush=True)

meta = None
for dp, dn, fn in os.walk('/kaggle/input'):
    if 'scene.json' in fn and 'log.json' in fn and 'trainval' in dp:
        meta = dp; break
assert meta, 'v1.0-trainval metadata not found'
ROOT = str(Path(meta).parent)
clock(f'metadata: {meta}')
clock(f'dataroot: {ROOT}')

In [ ]:
def load(name): return json.load(open(os.path.join(meta, name + '.json')))
scenes = load('scene'); logs = {l['token']: l for l in load('log')}
samples = load('sample'); sample_data = load('sample_data')
clock(f'{len(scenes)} scenes, {len(samples)} samples, {len(sample_data)} sample_data')

# --- reproduce labsd.splits.partition_by_location (name-sorted, 60/40 per city) ---
bos, sg = [], []
for s in scenes:
    loc = logs[s['log_token']]['location']
    if loc.startswith('boston'): bos.append((s['name'], s['token']))
    elif loc.startswith('singapore'): sg.append((s['name'], s['token']))
bos.sort(); sg.sort()
def split(items, r=0.6):
    n = max(1, math.floor(len(items) * r)); n = min(n, len(items) - 1)
    return [t for _, t in items[:n]], [t for _, t in items[n:]]
bos_tr, bos_val = split(bos); sg_tr, sg_val = split(sg)
clock(f'boston_train={len(bos_tr)} boston_val={len(bos_val)} sg_train={len(sg_tr)} sg_val={len(sg_val)}')

# --- even stride, same rule as labsd.splits.cap_val_scenes ---
def stride(toks, k):
    if len(toks) <= k: return list(toks)
    st = len(toks) / k
    return [toks[i] for i in sorted({int(i * st) for i in range(k)})]

splits = {
    'boston_c1':        stride(bos_tr, 100),
    'boston_c23':       stride(bos_val, 100),
    'singapore_update': stride(sg_tr, 100),
    'singapore_test':   stride(sg_val, 50),   # identical to the paper's 50 test scenes
}
allsel = [t for v in splits.values() for t in v]
assert len(allsel) == len(set(allsel)) == 350, 'parts must be disjoint and total 350'
for k, v in splits.items(): clock(f'{k}: {len(v)} scenes')

In [ ]:
# --- select full-size CAM_FRONT keyframe images of the chosen scenes ---
scene_of_sample = {s['token']: s['scene_token'] for s in samples}
part_of_scene = {t: k for k, v in splits.items() for t in v}
chosen = [sd for sd in sample_data
          if sd['is_key_frame'] and sd['filename'].startswith('samples/CAM_FRONT/')
          and scene_of_sample.get(sd['sample_token']) in part_of_scene]
clock(f'CAM_FRONT keyframes selected: {len(chosen)}')

OUT = Path('/kaggle/working/nuscenes_subset350')
(OUT / 'samples' / 'CAM_FRONT').mkdir(parents=True, exist_ok=True)
shutil.copytree(meta, OUT / 'v1.0-trainval', dirs_exist_ok=True)
clock('metadata copied')

per_part = {k: 0 for k in splits}; missing = 0; nbytes = 0
for i, sd in enumerate(chosen):
    src = os.path.join(ROOT, sd['filename'])
    if not os.path.exists(src): missing += 1; continue
    dst = OUT / sd['filename']
    shutil.copy2(src, dst); nbytes += os.path.getsize(dst)
    per_part[part_of_scene[scene_of_sample[sd['sample_token']]]] += 1
    if i % 2000 == 0: clock(f'  copied {i}/{len(chosen)}')
clock(f'images copied: {sum(per_part.values())}, missing: {missing}')

In [ ]:
json.dump(splits, open(OUT / 'splits_350.json', 'w'), indent=1)
def du(p): return sum(f.stat().st_size for f in Path(p).rglob('*') if f.is_file())
summary = {
    'scenes_per_part': {k: len(v) for k, v in splits.items()},
    'images_per_part': per_part,
    'images_total': sum(per_part.values()),
    'missing_images': missing,
    'images_GB': round(nbytes / 1e9, 3),
    'metadata_GB': round(du(OUT / 'v1.0-trainval') / 1e9, 3),
    'total_GB': round(du(OUT) / 1e9, 3),
}
json.dump(summary, open(OUT / 'subset_summary.json', 'w'), indent=1)
print(json.dumps(summary, indent=1))
clock('=== SUBSET DONE ===')